# LLM vs Traditional Sentiment Models — Climate News Classification

Compare Llama-3.1-8B (few-shot CoT) against VADER and TextBlob on the task of classifying climate news paragraphs as **favorable** or **unfavorable** to climate-friendly policies.

## 1. Setup & Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

import nltk
nltk.download('vader_lexicon', quiet=True)
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob

%matplotlib inline

LABEL_NAMES = ['favorable', 'unfavorable']

## 2. Configuration

In [ ]:
TRAIN_FILE = '/data/home/xiong/data/Fund/Climate/training_eval_results/Llama-3.1-8B-Instruct_long_fewshot_cot_train_results_v2.csv'
VAL_FILE = '/data/home/xiong/data/Fund/Climate/training_eval_results/Llama-3.1-8B-Instruct_long_fewshot_cot_val_results_v2.csv'
OUTPUT_DIR = Path('/data/home/xiong/data/Fund/Climate/llama_vs_traditional_method')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VADER_THRESHOLD = 0.0
TEXTBLOB_THRESHOLD = 0.0
NEUTRAL_STRATEGY = 'favorable'  # maps exact-zero / neutral -> favorable (project convention)

## 3. Helper Functions

In [ ]:
def load_data(file_path: str) -> pd.DataFrame:
    """Load and validate a results CSV file."""
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {path}")
    df = pd.read_csv(path)
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])
    required_cols = {'paragraph', 'true_label', 'predicted_label'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {path.name}: {missing}")
    return df


def predict_vader(texts, threshold=0.0, neutral_strategy='favorable'):
    sid = SentimentIntensityAnalyzer()
    predictions = []
    for text in texts:
        if not isinstance(text, str) or len(text.strip()) == 0:
            predictions.append(neutral_strategy)
            continue
        compound = sid.polarity_scores(text)['compound']
        if compound > threshold:
            predictions.append('favorable')
        elif compound < threshold:
            predictions.append('unfavorable')
        else:
            predictions.append(neutral_strategy)
    return predictions


def predict_textblob(texts, threshold=0.0, neutral_strategy='favorable'):
    predictions = []
    for text in texts:
        if not isinstance(text, str) or len(text.strip()) == 0:
            predictions.append(neutral_strategy)
            continue
        polarity = TextBlob(text).sentiment.polarity
        if polarity > threshold:
            predictions.append('favorable')
        elif polarity < threshold:
            predictions.append('unfavorable')
        else:
            predictions.append(neutral_strategy)
    return predictions


def clean_llama_predictions(labels, neutral_strategy='favorable'):
    mapping = {'favorable': 'favorable', 'unfavorable': 'unfavorable', 'neutral': neutral_strategy}
    cleaned = []
    for label in labels:
        if pd.isna(label):
            cleaned.append(neutral_strategy)
        else:
            cleaned.append(mapping.get(str(label).strip().lower(), neutral_strategy))
    return cleaned


def compute_metrics(y_true, y_pred):
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }
    for label in LABEL_NAMES:
        metrics[f'precision_{label}'] = precision_score(y_true, y_pred, labels=[label], average='micro', zero_division=0)
        metrics[f'recall_{label}'] = recall_score(y_true, y_pred, labels=[label], average='micro', zero_division=0)
        metrics[f'f1_{label}'] = f1_score(y_true, y_pred, labels=[label], average='micro', zero_division=0)
    metrics['classification_report'] = classification_report(y_true, y_pred, labels=LABEL_NAMES, zero_division=0)
    metrics['confusion_matrix'] = confusion_matrix(y_true, y_pred, labels=LABEL_NAMES)
    return metrics

## 4. Load Data

In [ ]:
train_df = load_data(TRAIN_FILE)
val_df = load_data(VAL_FILE)

datasets = {'train': train_df, 'val': val_df}

for name, df in datasets.items():
    n_fav = (df['true_label'] == 'favorable').sum()
    n_unfav = (df['true_label'] == 'unfavorable').sum()
    print(f"{name}: {len(df)} samples ({n_fav} favorable, {n_unfav} unfavorable)")

train_df.head()

In [ ]:
# Inspect label distributions
print("Train true_label value counts:")
print(train_df['true_label'].value_counts())
print("\nTrain predicted_label value counts (Llama raw):")
print(train_df['predicted_label'].value_counts())
print("\nVal predicted_label value counts (Llama raw):")
print(val_df['predicted_label'].value_counts())

## 5. Run Predictions (VADER, TextBlob, Llama)

In [ ]:
all_results = {}
all_model_preds = {}  # {split: {model: preds}}

for split_name, df in datasets.items():
    texts = df['paragraph']
    y_true = df['true_label'].tolist()

    llama_preds = clean_llama_predictions(df['predicted_label'], neutral_strategy=NEUTRAL_STRATEGY)
    vader_preds = predict_vader(texts, threshold=VADER_THRESHOLD, neutral_strategy=NEUTRAL_STRATEGY)
    textblob_preds = predict_textblob(texts, threshold=TEXTBLOB_THRESHOLD, neutral_strategy=NEUTRAL_STRATEGY)

    model_preds = {
        'Llama-3.1-8B': llama_preds,
        'VADER': vader_preds,
        'TextBlob': textblob_preds,
    }
    all_model_preds[split_name] = model_preds

    split_results = {}
    for model_name, preds in model_preds.items():
        split_results[model_name] = compute_metrics(y_true, preds)

    all_results[split_name] = split_results

print("Predictions complete for all splits.")

## 6. Classification Reports

In [ ]:
for split_name, split_results in all_results.items():
    for model_name, metrics in split_results.items():
        print(f"\n{'='*60}")
        print(f"  {model_name} — {split_name} set")
        print(f"{'='*60}")
        print(metrics['classification_report'])

## 7. Comparison Summary Table

In [ ]:
metric_keys = [
    'accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
    'precision_favorable', 'recall_favorable', 'f1_favorable',
    'precision_unfavorable', 'recall_unfavorable', 'f1_unfavorable',
]
rows = []
for split_name, models in all_results.items():
    for model_name, metrics in models.items():
        row = {'split': split_name, 'model': model_name}
        for key in metric_keys:
            row[key] = metrics.get(key)
        rows.append(row)

comparison_df = pd.DataFrame(rows).sort_values(['split', 'f1_macro'], ascending=[True, False])
comparison_df.round(4)

In [ ]:
# Save metrics comparison
comparison_df.to_csv(OUTPUT_DIR / 'metrics_comparison.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'metrics_comparison.csv'}")

## 8. Disagreement Analysis

In [ ]:
all_disagreements = {}

for split_name, df in datasets.items():
    model_preds = all_model_preds[split_name]
    result = df[['paragraph', 'true_label']].copy()
    if 'justification' in df.columns:
        result['justification'] = df['justification']

    for model_name, preds in model_preds.items():
        result[f'pred_{model_name}'] = preds

    pred_cols = [f'pred_{m}' for m in model_preds.keys()]
    result['all_agree'] = result[pred_cols].nunique(axis=1) == 1

    def get_correct_models(row):
        correct = [m for m in model_preds if row[f'pred_{m}'] == row['true_label']]
        return ', '.join(correct) if correct else 'none'

    result['correct_models'] = result.apply(get_correct_models, axis=1)
    all_disagreements[split_name] = result

    # Save
    dis_path = OUTPUT_DIR / f'disagreements_{split_name}.csv'
    result.to_csv(dis_path, index=False)
    n_disagree = (~result['all_agree']).sum()
    print(f"{split_name}: {n_disagree}/{len(result)} samples with disagreement -> {dis_path}")

In [ ]:
# Inspect disagreement samples (val set)
val_disagree = all_disagreements['val']
val_disagree[~val_disagree['all_agree']].head(10)

## 9. LLM Advantage Cases

Rows where Llama predicted correctly but **both** VADER and TextBlob got it wrong — showcasing the LLM's contextual understanding.

In [ ]:
llm_advantage = {}

for split_name, dis_df in all_disagreements.items():
    llama_correct = dis_df['pred_Llama-3.1-8B'] == dis_df['true_label']
    vader_wrong = dis_df['pred_VADER'] != dis_df['true_label']
    textblob_wrong = dis_df['pred_TextBlob'] != dis_df['true_label']

    adv_df = dis_df[llama_correct & vader_wrong & textblob_wrong].copy()
    llm_advantage[split_name] = adv_df

    adv_path = OUTPUT_DIR / f'llm_advantage_{split_name}.csv'
    adv_df.to_csv(adv_path, index=False)
    print(f"{split_name}: {len(adv_df)} LLM advantage samples -> {adv_path}")

In [ ]:
# Inspect LLM advantage examples (val set)
llm_advantage['val'].head(10)

## 10. Confusion Matrices

In [ ]:
for split_name, split_results in all_results.items():
    model_names = list(split_results.keys())
    n_models = len(model_names)

    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
    if n_models == 1:
        axes = [axes]

    for ax, model_name in zip(axes, model_names):
        cm = split_results[model_name]['confusion_matrix']
        ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        ax.set_title(f'{model_name}\n({split_name} set)', fontsize=12)
        ax.set_xticks(range(len(LABEL_NAMES)))
        ax.set_yticks(range(len(LABEL_NAMES)))
        ax.set_xticklabels(LABEL_NAMES, rotation=45, ha='right')
        ax.set_yticklabels(LABEL_NAMES)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        thresh = cm.max() / 2.0
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center',
                        color='white' if cm[i, j] > thresh else 'black')

    fig.suptitle(f'Confusion Matrices \u2014 {split_name.capitalize()} Set', fontsize=14, y=1.02)
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / f'confusion_matrices_{split_name}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 11. Metrics Comparison Bar Charts

In [ ]:
macro_metrics = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
display_names = ['Accuracy', 'Precision', 'Recall', 'F1']
splits = comparison_df['split'].unique()
models = comparison_df['model'].unique()
colors = ['#2196F3', '#FF9800', '#4CAF50']

fig, axes = plt.subplots(1, len(splits), figsize=(8 * len(splits), 6), sharey=True)
if len(splits) == 1:
    axes = [axes]

for ax, split in zip(axes, splits):
    split_data = comparison_df[comparison_df['split'] == split]
    x = np.arange(len(macro_metrics))
    width = 0.25
    for i, (_, row) in enumerate(split_data.iterrows()):
        values = [row[m] for m in macro_metrics]
        offset = (i - len(models) / 2 + 0.5) * width
        bars = ax.bar(x + offset, values, width, label=row['model'], color=colors[i % len(colors)])
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8)
    ax.set_title(f'{split.capitalize()} Set', fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(display_names, fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Model Comparison: Macro Metrics', fontsize=14)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-class F1
fig, axes = plt.subplots(1, len(splits), figsize=(8 * len(splits), 5), sharey=True)
if len(splits) == 1:
    axes = [axes]

for ax, split in zip(axes, splits):
    split_data = comparison_df[comparison_df['split'] == split]
    x = np.arange(len(models))
    width = 0.35
    f1_fav = split_data['f1_favorable'].values
    f1_unfav = split_data['f1_unfavorable'].values
    model_labels = split_data['model'].values

    ax.bar(x - width / 2, f1_fav, width, label='favorable', color='#4CAF50')
    ax.bar(x + width / 2, f1_unfav, width, label='unfavorable', color='#f44336')
    ax.set_title(f'{split.capitalize()} Set', fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels, fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('F1 Score')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Per-Class F1 Scores by Model', fontsize=14)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'f1_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Summary

In [ ]:
display_cols = ['split', 'model', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
print("=" * 80)
print("  MODEL COMPARISON SUMMARY: LLM vs Traditional Sentiment Models")
print("=" * 80)
print(comparison_df[display_cols].round(4).to_string(index=False))
print("=" * 80)
print("\n  LLM Advantage Cases (Llama correct, VADER+TextBlob both wrong):")
for split_name, adv_df in llm_advantage.items():
    print(f"    {split_name}: {len(adv_df)} samples")
print("=" * 80)